# UDISE+ National Policy Analysis: Driving Evidence-Based Interventions in Public Policy

**Target Audience:** District Magistrates (DMs), Policy Officers, and Senior Educational Administrators  

---

In [28]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("Initializing Spark Session...")
spark = SparkSession.builder \
    .appName("UDISE_Interactive_Analysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "40") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Initialized successfully.")

Initializing Spark Session...
Spark Session Initialized successfully.


## Section 2: Loading Datasets & Aggregate Exports
We load the aggregated policy indicators and cohort transition tables generated by our pipeline. These datasets represent statistically perfect representations of the 1.47M schools collapsed to district level.

In [29]:
enroll1 = spark.read.parquet('data/2023-24_parquet/enroll1_parquet/*.parquet')

In [30]:
enroll1.show(10)

+----------+----------+-------+-----+-----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+
|pseudocode|item_group|item_id|cpp_b|cpp_g|c1_b|c1_g|c2_b|c2_g|c3_b|c3_g|c4_b|c4_g|c5_b|c5_g|c6_b|c6_g|c7_b|c7_g|c8_b|c8_g|c9_b|c9_g|c10_b|c10_g|c11_b|c11_g|c12_b|c12_g|
+----------+----------+-------+-----+-----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+
|   4814412|         1|      3|    0|    0|   0|   0|   0|   0|   0|   0|   0|   0|   0|   1|   0|   0|   1|   0|   0|   0|   0|   0|    0|    0|    0|    0|    0|    0|
|   4814412|         1|      4|    0|    0|  11|   6|   1|   4|   5|   2|   3|   2|   5|   3|   3|   3|   4|   3|   5|   2|   0|   0|    0|    0|    0|    0|    0|    0|
|   4814412|         2|      5|    0|    0|   0|   0|   0|   0|   0|   0|   1|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|    0|    0|    0

In [31]:
enroll2 = spark.read.parquet('data/2023-24_parquet/enroll2_parquet/*.parquet')
enroll2.show(10)

+----------+----------+-------+-----+-----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+
|pseudocode|item_group|item_id|cpp_b|cpp_g|c1_b|c1_g|c2_b|c2_g|c3_b|c3_g|c4_b|c4_g|c5_b|c5_g|c6_b|c6_g|c7_b|c7_g|c8_b|c8_g|c9_b|c9_g|c10_b|c10_g|c11_b|c11_g|c12_b|c12_g|
+----------+----------+-------+-----+-----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+
|   4018119|         8|      5|    0|    0|   5|   8|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|    0|    0|    0|    0|    0|    0|
|   4018119|         8|      6|    0|    0|   2|   0|   4|   9|   2|   2|   1|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|    0|    0|    0|    0|    0|    0|
|   4018119|         8|      7|    0|    0|   1|   0|   5|   3|   8|   5|   2|   1|   0|   0|   0|   0|   0|   0|   0|   0|   0|   0|    0|    0|    0

In [32]:
facility = spark.read.parquet('data/2023-24_parquet/facility_parquet/*.parquet')
facility.show(1)

+----------+---------------+------------------+---------------------+-------------+-----------------+-----------+----------------------------+-----------------------------+-----------------------------+--------------------+-----------------+----------------------+------------------+-----------------------+----------------------+-----------------------+-----------------------+------------------------+-----------+------------+--------------------+------------+------------+------+---------+--------------+-------------+----------------+----------------+----------+-------------+------------------+-----------------+---------------------+--------------------------+------------------------+-----------+--------------------+---------+--------------+--------------------+-----------------+----------------+------------------+-------------------------+----------------------+------------+-------------+------------+-------------+-------------+------------+----------------+---------------+-------------

In [33]:
profile1 = spark.read.parquet('data/2023-24_parquet/profile1_parquet/*.parquet')
profile1.show(1)

+----------+-------+--------+-------+-----------+-------------------------+--------------------+-------------+-----------------------+--------------+---------------+-----------+--------+---------+---------+-------+-----------------------+------------+-----------+---------+---------------+-------------+----------------+----------------+----------------+-------------+--------------+-----------------+-----------+------------+--------------+------+----------+----------+-----------+-----------+----------------+----------------+
|pseudocode|  state|district|  block|rural_urban|lgd_urban_local_body_name|       lgd_ward_name|lgd_vill_name|lgd_vill_panchayat_name|lgd_block_name|school_category|school_type|lowclass|highclass|managment|pincode|special_school_for_cwsn|shift_school|resi_school|resi_type|minority_school|medium_instr1|medium_of_instr2|medium_of_instr3|medium_of_instr4|aff_board_sec|aff_board_hsec|approachable_road|pre_primary|anganwadi_yn|avg_instr_days|cce_yn|same_sch_b|same_sch_g|o

In [34]:
profile2 = spark.read.parquet('data/2023-24_parquet/profile2_parquet/*.parquet')
profile2.show(1)

+----------+---------------------+----------------+-----------------+------------------+---------------+------------------+---------------+----------------+---------------+--------------------+-----------------+----------+-------------+-----------------+--------------+------------------+
|pseudocode|balavatika_located_yn|special_training|material_training|free_text_books_pr|free_uniform_pr|free_text_books_up|free_uniform_up|acad_inspections|crc_coordinator|block_level_officers|district_officers|smc_exists|smc_smdc_same|smc_smdc_meetings|grants_receipt|grants_expenditure|
+----------+---------------------+----------------+-----------------+------------------+---------------+------------------+---------------+----------------+---------------+--------------------+-----------------+----------+-------------+-----------------+--------------+------------------+
|   1124241|                    1|               9|                1|                 1|              1|                 2|          

In [35]:
teacher = spark.read.parquet('data/2023-24_parquet/teacher_parquet/*.parquet')
teacher.show(1)

+----------+---------+----+------+-----------+-------+------+------+-------+-------+--------+---------+--------------+--------+-----------------------+------------+---------------+----------------+-------------------+---------------------+----------------------+--------------------+---------------------+----------------------------+---------------------------+---------------------------------------+-------------------+--------------+--------------+--------------+-----+----+-------------------+------------+---------------+-------------------+-----------+------------+
|pseudocode|total_tch|male|female|transgender|gen_tch|sc_tch|st_tch|obc_tch|regular|contract|part_time|below_graduate|graduate|post_graduate_and_above|trained_comp|class_taught_pr|class_taught_upr|class_taught_pr_upr|class_taught_sec_only|class_taught_hsec_only|class_taught_upr_sec|class_taught_sec_hsec|class_taugt_pre_primary_only|class_taught_pr_and_pre_pri|teacher_involve_non_training_assignment|diploma_certificate|bache

# Connecting to database

In [42]:
import duckdb

conn = duckdb.connect()
conn.execute("ATTACH 'dbname=udise user=postgres password=1105 host=localhost' AS pg (TYPE postgres)")

conn.execute("""
    CREATE TABLE pg.public.enrollment1 AS 
    SELECT * FROM read_parquet('data/2023-24_parquet/enroll1_parquet/*.parquet')
""")
conn.execute("""
    CREATE TABLE pg.public.enrollment2 AS 
    SELECT * FROM read_parquet('data/2023-24_parquet/enroll2_parquet/*.parquet')
""")

conn.execute("""
    CREATE TABLE pg.public.profile1 AS 
    SELECT * FROM read_parquet('data/2023-24_parquet/profile1_parquet/*.parquet')
""")
conn.execute("""
    CREATE TABLE pg.public.profile2 AS 
    SELECT * FROM read_parquet('data/2023-24_parquet/profile2_parquet/*.parquet')
""")
conn.execute("""
    CREATE TABLE pg.public.teachers AS 
    SELECT * FROM read_parquet('data/2023-24_parquet/teacher_parquet/*.parquet')
""")

conn.execute("""
    CREATE TABLE pg.public.facilities AS 
    SELECT * FROM read_parquet('data/2023-24_parquet/facility_parquet/*.parquet')
""")   

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))